# PHASE: W9-F — Frozen E3-k3 utilization

**SCIENTIFIC QUESTION:** Given identical compact E3-k3 evidence, should the generator regenerate, conservatively revise, or use evidence-anchored revision?

**PREREQUISITES:** Completed W9-E3 FIT surface, selected k=3, immutable A–E3 lineage.

**INPUT ARTIFACTS:** `configs/week9/w9f_utilization.yaml`; saved E3-k3 evidence/results and D0.

**EXPECTED OUTPUTS:** separate `results/week9/w9f_utilization/v1` protocol, prompts, alias proof, quality/cost/transition/rescue/bootstrap tables, F2/action freezes and FIT-only controller handoff.

**CURRENT SPLIT:** FIT only. No DEV/INTERNAL-VAL/TEST tuning.

**FULL RUN FLAG:** `RUN_F_FULL = False`; `RUN_F_SMOKE = False`. Run All defaults to read-only inspection, without loading the generator. W9-G remains OFF.

## 1. Protocol

The only intervention variable is the answer interface. Frozen FIXED-5 / SEMANTIC@20 / DraftSemantic coarse K=2, query-only MiniLM top-3 sentences, and source-order presentation remain identical. No retrieval, sentence rescoring, k changes or new generator. D0 is SKIP; F0 is fresh regeneration and is a separate action.


In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/week9').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open within CS787-RAG-Project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.week9.provenance import branch_guard
from src.week9.utilization_f.protocol import config
from src.week9.utilization_f.prompts import definitions
branch_guard()
RUN_F_FULL = True
RUN_F_SMOKE = False
SMOKE_LIMIT = 2
# Legacy full-run switch is disabled; only RUN_F_* controls this notebook.
RUN_FULL = False
if RUN_FULL:
    raise ValueError('Use the explicit RUN_F_FULL switch for the new E3-k3 protocol')
CONFIG = ROOT / 'configs/week9/w9f_utilization.yaml'
cfg = config(CONFIG)
RESULT_ROOT = ROOT / cfg['output_root']
SPLIT = 'fit'
def show_json(name):
    path = RESULT_ROOT / name
    if path.exists():
        display(json.loads(path.read_text()))
    else:
        print('Not available yet:', name)
def show_csv(name):
    path = RESULT_ROOT / ('fit' if (RESULT_ROOT/'fit/manifest.json').exists() else 'smoke') / name
    if path.exists():
        display(pd.read_csv(path))
    else:
        print('Not available yet:', name)


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Frozen dependencies

The runner verifies existing A–E3 protocols, code/config/model fingerprints, row checksums and completed FIT coverage. It never writes old results. `--stage prepare --resume` performs CPU-only identity verification and pins W9-F prompts before generation; smoke/full also do this automatically.


In [2]:
show_json('protocol.json')


{'schema': 'W9-F-E3k3-utilization-v1',
 'config': {'experiment': 'W9-F-E3k3-utilization-v1',
  'output_root': 'results/week9/w9f_utilization/v1',
  'e3_config': 'configs/week9/w9e3_senttopk.yaml',
  'split': 'fit',
  'sentence_k': 3,
  'policies': ['F0', 'F1', 'F2-A', 'F2-B'],
  'f2_variants': ['F2-A', 'F2-B'],
  'max_new_tokens': 128,
  'seed': 42,
  'smoke_queries': 2,
  'quality_tolerance': 0.003,
  'material_hurt_difference': 0.01,
  'bootstrap_resamples': 10000,
  'train_controller': False},
 'split': 'fit',
 'sources': {'src/week9/utilization_f/__init__.py': 'a1525c7f0e4c2e9ed0c1100f76bf8dfc0236d271338af6090abc9706e0bd43de',
  'src/week9/utilization_f/analysis.py': 'b4fe72011333cd4cb5fa0549c312e02e1b4e3da05c260c21e1351feddc90c3b8',
  'src/week9/utilization_f/prompts.py': 'a28210fa5e1b6b4ad51ba59da2c071e0b7b2b1d424115ad65dae674c85543feb',
  'src/week9/utilization_f/protocol.py': 'd8309dd4dbe8145d9af57c95cbb40b6a2128c4a0b08f40bb8e9b28c607dea011',
  'src/week9/utilization_f/runner.p

## 3. Load E3-k3

Evidence is loaded from saved rows, not selected again. The inventory contains exact sentence text, IDs, source hashes and character offsets.


In [3]:
e3_root = ROOT / 'results/week9/w9e3_senttopk/v1'
freeze = json.loads((e3_root / 'frozen_e3_senttopk.json').read_text())
assert freeze['selected_k'] == 3 and freeze['selection_split'] == 'fit'
display({'selected_k': freeze['selected_k'], 'primary_policy': freeze['primary_policy_unchanged']})


{'selected_k': 3, 'primary_policy': 'K_E=0'}

## 4. Source/evidence identity assertions

Question/paper, coarse IDs and content hashes, selected sentence IDs and exact text hashes, E3 protocol/config identity, rendered F0 prompts and source order must match. A mismatch fails loudly without substitutions.


In [4]:
identity_path = RESULT_ROOT/'source_evidence_identities.json'
if identity_path.exists():
    display(pd.DataFrame(json.loads(identity_path.read_text())).head())
else:
    print('Run prepare or the explicit smoke to produce the identity proof.')


,question_id,paper_id,source_row_hash,evidence_hash,source_chunk_ids,source_chunk_hashes,selected_sentence_ids,sentence_text_hashes
0,512,139,c6bf7996dc72e6920e095c60565c9537592fedf0fb07f9...,10c49738e9f4234c7441f2090146b0848a56ed3d4b3ff6...,"[6ad011395e54b7dc9a41e8da, 1709dce1435dd7d6205...",{'6ad011395e54b7dc9a41e8da': 'a2bd819a90f48702...,"[0738e926e89c2b2b2751f2dc, c0832240e1008a5cd31...",[1f77f7d747b0c7411323c5ac0e71e4946a47477420a82...
1,513,139,7428854f74c51e10e29c5c80bb5df0e3206a9f692b2bbb...,e7bfb1e96b2c6cedfbd4f9ae906b81053e291ba8d1323e...,"[a1ae67b60f73b96ba3b358e1, 6ad011395e54b7dc9a4...",{'a1ae67b60f73b96ba3b358e1': 'f420ba10561dadee...,"[e03cb75da91a647583fe567f, 0738e926e89c2b2b275...",[73a9dfd198277f6bb673d63310d63fbc2b37405eeb7cf...
2,745,207,a19735b05cc71a36ebb10bccaee3419466ebdd7b9eeec2...,c1bd14eb2fe369bcd24aea7406b65598e92c811428fc55...,"[40472a34da2201033d10df7d, d7854006fc2f6cd57fe...",{'40472a34da2201033d10df7d': '4a5ce5389ca60137...,"[f109542404dc355c665185fe, 7954b5baac886761b39...",[cc350201a1ef6c23b39d3b0eb077c87793f959edce178...
3,746,207,2e4c4a86e01bca59954e3a6590ed90ab480c5c9865c63b...,d2a906bc82dfc0131299cda65fa75cabdf5f63e9fe71be...,"[f80afaa77efd1ca0d6633daa, d7854006fc2f6cd57fe...",{'f80afaa77efd1ca0d6633daa': 'b06a7e8068b6d489...,"[22ffa51b66be318847d5bf2f, a7349c521eb1bea8307...",[8d8a82a9cf8b99ef0a8a775e57a9e0a68d98e4dbbb65e...
4,1633,521,70104e121f1af8f107b2a53aaa6964334f771f951cc9b9...,db368615faa3226f8e9bc8f44ec8e296c93ae63f76e830...,"[ccf012b296d6bfab6888d925, b83d44303a90da04131...",{'ccf012b296d6bfab6888d925': '74470f821c48e968...,"[50e0ce49d32836c6f5f1c57d, c0c040ddb43f2e3755f...",[129542c7ad2807be779c53fcd633f3151c78a3f7ce859...


## 5. D0 — keep original draft

D0 loads the original first-pass answer. It has no second answer call. It is not a candidate for freezing an intervention mode; it remains the later external SKIP action.

## 6. F0 alias/reproduction verification

Saved E3-k3 is reused only after exact generator/checkpoint/tokenizer, canonical context/latent-slot identity, final prompt hash, evidence formatting, greedy decoding/128 cap, generation sources and saved LM-call/output checks. No new F0 generation is needed when this proof passes.


In [5]:
proof_path = RESULT_ROOT/'f0_alias_verification.json'
if proof_path.exists():
    proof = json.loads(proof_path.read_text())
    display({k:v for k,v in proof.items() if k != 'proofs'})
else:
    print('Alias proof pending CPU preparation; no F0 generation will be silently substituted.')


{'equivalent': True,
 'queries': 1746,
 'upstream_protocol_hash': '33ac5ee8700b85524609a40fc433a2a32bfa0ec248095e03e99872cc8f016f76',
 'model': {'model_id': 'mistralai/Mistral-7B-Instruct-v0.2',
  'model_revision': 'b52c0e84decd72f68ff7fcaad4d9a5f8299d7115a535c077bc05df5afda6005b',
  'dtype': 'torch.float16',
  'quantization': 'none',
  'interface': 'trained_SARA',
  'attention_backend': 'efficient_sdpa'},
 'checks': ['source/config/model code unchanged',
  'checkpoint/tokenizer identities',
  'rendered final prompt hash',
  'sentence/evidence hashes',
  'greedy decoding/128 cap',
  'matching saved LM call/output',
  'canonical latent context and initial token count']}

## 7. F1 — conservative revision

Preserve the original answer unless the evidence clearly requires a minimal correction/addition. One fixed safety/preservation prompt, not tuned on W9-F outcomes.


In [6]:
display(definitions()['F1'])


{'messages': [{'role': 'user',
   'content': 'The initial answer may already be correct. Review the additional source evidence. Preserve the original answer unless the evidence clearly requires a correction or addition. Make the smallest necessary change. Do not add unsupported information. Use only the supplied source evidence for any correction or addition. Output only the final concise answer, without reasoning or chain-of-thought.\n\nInitial answer: {initial_draft}\n\nAdditional evidence:\n{evidence}\n\nQuestion: {question}'}],
 'prompt_hash': 'e692aea57c2bb3c0e3d3b12cf862230f91ff3463ff0b32dd94a0ec75affff201'}

## 8. F2 — two predeclared evidence-anchored variants

F2-A is direct evidence-authoritative revision. F2-B explicitly considers missing information, contradictions and supported content. Both output only the answer, never reasoning. The exact templates and hashes are pinned before the first call. No iterative prompt search.


In [7]:
display({p:d for p,d in definitions().items() if p.startswith('F2-')})


{'F2-A': {'messages': [{'role': 'user',
    'content': 'Answer the question using the supplied source evidence. Treat the source evidence as authoritative. The initial answer is only a starting point and may be incomplete or wrong. Preserve parts of the initial answer only when they remain supported by the evidence. Add missing information supported by the evidence, correct conflicting claims, and remove unsupported material when necessary. Do not invent facts. Output only the final concise answer, without reasoning or chain-of-thought.\n\nInitial answer: {initial_draft}\n\nAdditional evidence:\n{evidence}\n\nQuestion: {question}'}],
  'prompt_hash': '10b48d1969224dbf0f5d79d3bfd7844ab828f44e0ae533f77b8d5f7f16467795'},
 'F2-B': {'messages': [{'role': 'user',
    'content': 'Answer the question using the supplied source evidence as the authoritative source. Use the initial answer as a starting point, not as an authority; it may be incomplete or wrong. Consider information missing from th

## 9. Tiny deterministic FIT smoke / 10. Full FIT generation

Choose one explicit flag. The smoke uses two deterministic FIT queries, aliases D0/F0 and makes three new revision calls/query. It does not select or freeze anything. Full generation is manually enabled only. Up to 5,238 new answer calls; F0 is reused. Resume preserves independently completed LM calls and result rows.


In [8]:
assert not (RUN_F_FULL and RUN_F_SMOKE), 'Choose smoke or full'
if RUN_F_FULL or RUN_F_SMOKE:
    stage = 'full' if RUN_F_FULL else 'smoke'
    command = [str(ROOT/'sara_env/bin/python'), '-m', 'src.week9.utilization_f.runner',
               '--config', str(CONFIG), '--stage', stage, '--resume']
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('W9-F generation disabled. Saved artifacts are inspected only.')


Loading unchanged SARA for revision only. No extraction LM/encoder/controller.


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.36it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `m

W9-F full: 1/1746 query 512, F1
W9-F full: 1/1746 query 512, F2-A
W9-F full: 1/1746 query 512, F2-B
W9-F full: 2/1746 query 513, F1
W9-F full: 2/1746 query 513, F2-A
W9-F full: 2/1746 query 513, F2-B
W9-F full: 3/1746 query 745, F1
W9-F full: 3/1746 query 745, F2-A
W9-F full: 3/1746 query 745, F2-B
W9-F full: 4/1746 query 746, F1
W9-F full: 4/1746 query 746, F2-A
W9-F full: 4/1746 query 746, F2-B
W9-F full: 5/1746 query 1633, F1
W9-F full: 5/1746 query 1633, F2-A
W9-F full: 5/1746 query 1633, F2-B
W9-F full: 6/1746 query 1221, F1
W9-F full: 6/1746 query 1221, F2-A
W9-F full: 6/1746 query 1221, F2-B
W9-F full: 7/1746 query 1222, F1
W9-F full: 7/1746 query 1222, F2-A
W9-F full: 7/1746 query 1222, F2-B
W9-F full: 8/1746 query 1223, F1
W9-F full: 8/1746 query 1223, F2-A
W9-F full: 8/1746 query 1223, F2-B
W9-F full: 9/1746 query 56, F1
W9-F full: 9/1746 query 56, F2-A
W9-F full: 9/1746 query 56, F2-B
W9-F full: 10/1746 query 57, F1
W9-F full: 10/1746 query 57, F2-A
W9-F full: 10/1746 query 

In [9]:
smoke_path = RESULT_ROOT/'smoke/results.jsonl'
if smoke_path.exists():
    from src.week9.caching import read_jsonl
    smoke = read_jsonl(smoke_path)
    for q in dict.fromkeys(r['question_id'] for r in smoke):
        group = [r for r in smoke if r['question_id'] == q]
        f0 = next(r for r in group if r['policy'] == 'F0')
        assert len({r['evidence_hash'] for r in group if r['policy'] != 'D0'}) == 1
        display({'question_id': q, 'question': f0['question'], 'y0': f0['initial_draft'],
                 'exact_evidence': f0['evidence_text'], 'sentence_ids': f0['selected_sentence_ids']})
        display(pd.DataFrame([{k:r.get(k) for k in ['policy','prediction','raw_generation','answer_fail_closed','f1']} for r in group]))
else:
    print('No smoke results yet.')


{'question_id': '218',
 'question': 'Did the authors use crowdsourcing platforms?',
 'y0': 'Yes',
 'exact_evidence': 'We paired Turkers at random and let them alternate between the tourist and guide role across different HITs.\n\nTurkers successfully completed INLINEFORM0 of all finished tasks (we use this statistic as the human success rate).\n\nTurkers use diverse discourse, with a vocabulary size of more than 10K (calculated over all successful dialogues).',
 'sentence_ids': ['808c3dd318b0bad0f74b86bb',
  '91328a7552ae527a7df1cf81',
  'a1392154638a17edddfff85f']}

,policy,prediction,raw_generation,answer_fail_closed,f1
0,D0,Yes,NaN,False,1.0
1,F0,Yes,NaN,False,1.0
2,F1,Yes,Yes,False,1.0
3,F2-A,Yes,Yes,False,1.0
4,F2-B,Yes,Yes,False,1.0


{'question_id': '1876',
 'question': "Do the authors give any examples of major events which draw the public's attention and the impact they have on stock price?",
 'y0': 'No',
 'exact_evidence': 'or its stock won’t override the overall stock market or sector’s impact on the stock return.\n\nPredicting stock price direction using Twitter sentiment is challenging but promising.\n\nWhich stock and what to predict is more important than how to predict.',
 'sentence_ids': ['6ad2aa47a339610a207c97b1',
  '529feaee3ecbbe4edfd5fd9b',
  'f18c600c3d308e7f1a85704d']}

,policy,prediction,raw_generation,answer_fail_closed,f1
0,D0,No,NaN,False,0.0
1,F0,Yes,NaN,False,1.0
2,F1,No,No,False,0.0
3,F2-A,No,No,False,0.0
4,F2-B,Yes,Yes,False,1.0


## 11. Quality metrics / 12. HELP and HURT

HELP/HURT thresholds are ±0.01 F1 vs D0. Report exact and normalized preservation, normalized intervention, answer/F1 differences vs F0 and mean positive/negative change magnitudes. HELP/(HELP+HURT) is descriptive.


In [10]:
show_csv('summary.csv')


,policy,n_queries,f1,exact_match,delta_f1,delta_f1_vs_f0,help,hurt,answer_changed,strict_preservation_rate,...,f0_hurt_queries,f0_help_queries,partial_rescue_rate,full_recovery_rate,hurt_rescue_rate,help_retention_rate,lost_gain_rate,diminished_gain_rate,actual_policy,post_selection
0,D0,1746,0.385631,0.227950,0.000000,-0.004393,0.000000,0.000000,0.000000,1.000000,...,365,378,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,NaN,NaN
1,F0,1746,0.390024,0.226231,0.004393,0.000000,0.216495,0.209049,0.613402,0.386598,...,365,378,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,NaN,NaN
2,F1,1746,0.387094,0.229095,0.001463,-0.002930,0.018328,0.014318,0.058992,0.941008,...,365,378,0.969863,0.950685,0.950685,0.071429,0.928571,0.952381,NaN,NaN
3,F2-A,1746,0.389616,0.231386,0.003985,-0.000408,0.021764,0.008591,0.056128,0.943872,...,365,378,0.972603,0.958904,0.958904,0.079365,0.920635,0.944444,NaN,NaN
4,F2-B,1746,0.386866,0.226231,0.001235,-0.003158,0.028064,0.017182,0.070447,0.929553,...,365,378,0.953425,0.936986,0.936986,0.103175,0.896825,0.928571,NaN,NaN
5,F2-frozen,1746,0.389616,0.231386,0.003985,-0.000408,0.021764,0.008591,0.056128,0.943872,...,365,378,0.972603,0.958904,0.958904,0.079365,0.920635,0.944444,F2-A,True


## 13. Transition, rescue and lost-gain analysis

Group by F0 HELP / HURT / approximately unchanged vs D0. Among F0-HURT queries: partial rescue is revision > F0+0.01; full recovery is revision ≥ D0−0.01; hurt rescue means no longer HURT. Among F0-HELP queries: HELP retention is still HELP, lost gain means no longer HELP, diminished gain is revision < F0−0.01. Empty denominators remain null. Labels never enter inference.


In [11]:
show_csv('transition_analysis.csv')
show_csv('rescue_analysis.csv')


,policy,f0_state,n_queries,mean_delta_vs_d0,mean_delta_vs_f0,preserve_rate,help_rate,hurt_rate,f0_hurt_queries,f0_help_queries,partial_rescue_rate,full_recovery_rate,hurt_rescue_rate,help_retention_rate,lost_gain_rate,diminished_gain_rate
0,D0,HELP,378,0.000000,-0.315075,1.000000,0.000000,0.000000,0,378,NaN,NaN,NaN,0.000000,1.000000,1.000000
1,D0,HURT,365,0.000000,0.305172,1.000000,0.000000,0.000000,365,0,1.000000,1.000000,1.000000,NaN,NaN,NaN
2,D0,UNCHANGED,1003,0.000000,0.000041,1.000000,0.000000,0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,F0,HELP,378,0.315075,0.000000,0.002646,1.000000,0.000000,0,378,NaN,NaN,NaN,1.000000,0.000000,0.000000
4,F0,HURT,365,-0.305172,0.000000,0.000000,0.000000,1.000000,365,0,0.000000,0.000000,0.000000,NaN,NaN,NaN
5,F0,UNCHANGED,1003,-0.000041,0.000000,0.697906,0.000000,0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN
6,F1,HELP,378,0.028217,-0.286857,0.880952,0.071429,0.007937,0,378,NaN,NaN,NaN,0.071429,0.928571,0.952381
7,F1,HURT,365,-0.020985,0.284187,0.923288,0.008219,0.049315,365,0,0.969863,0.950685,0.950685,NaN,NaN,NaN
8,F1,UNCHANGED,1003,-0.000452,-0.000411,0.975075,0.001994,0.003988,0,0,NaN,NaN,NaN,NaN,NaN,NaN
9,F2-A,HELP,378,0.031866,-0.283209,0.896825,0.079365,0.000000,0,378,NaN,NaN,NaN,0.079365,0.920635,0.944444


,policy,f0_hurt_queries,f0_help_queries,partial_rescue_rate,full_recovery_rate,hurt_rescue_rate,help_retention_rate,lost_gain_rate,diminished_gain_rate
0,D0,365,378,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000
1,F0,365,378,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
2,F1,365,378,0.969863,0.950685,0.950685,0.071429,0.928571,0.952381
3,F2-A,365,378,0.972603,0.958904,0.958904,0.079365,0.920635,0.944444
4,F2-B,365,378,0.953425,0.936986,0.936986,0.103175,0.896825,0.928571


## 14. Cost accounting

All policies conceptually include the initial draft. F1/F2 additionally expose y0 in their prompt. Report final/cumulative prompt/generated tokens, LM calls and stage latency. F0 costs are inherited from E3, not newly measured here. E3 embedding/index costs remain inherited and nonzero conceptually; no embedding is recomputed. No full end-to-end time is claimed with cached initial work.


In [12]:
show_csv('cost_summary.csv')
show_csv('lexical_diagnostics.csv')


,policy,n_queries,final_prompt_tokens,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls,evidence_tokens,retrieval_query_time,selection_latency,revision_latency,measured_end_to_end_latency,inherited_sentence_index_conceptual_seconds,inherited_query_embedding_conceptual_seconds,inherited_embedding_lookup_compute_latency,inherited_sentence_ranking_latency
0,D0,1746,940.544101,940.544101,16.654639,1.0,0.000000,0.00496,8.662111e-07,NaN,NaN,NaN,NaN,NaN,NaN
1,F0,1746,1085.768614,2026.312715,39.949026,2.0,98.332188,0.00496,2.936539e-01,1.022722,NaN,0.22256,0.007695,0.147695,0.001583
2,F1,1746,1157.446163,2097.990263,33.525200,2.0,98.332188,0.00496,2.936539e-01,0.854944,NaN,0.22256,0.007695,0.147695,0.001583
3,F2-A,1746,1179.446163,2119.990263,33.200458,2.0,98.332188,0.00496,2.936539e-01,0.848340,NaN,0.22256,0.007695,0.147695,0.001583
4,F2-B,1746,1190.446163,2130.990263,33.102520,2.0,98.332188,0.00496,2.936539e-01,0.847642,NaN,0.22256,0.007695,0.147695,0.001583


,question_id,paper_id,policy,answer_evidence_token_overlap,added_answer_token_count,removed_answer_token_count,added_tokens_in_evidence,added_token_evidence_fraction
0,512,139,D0,0.00,0,0,0,NaN
1,512,139,F0,1.00,3,0,3,1.0
2,512,139,F1,1.00,3,0,3,1.0
3,512,139,F2-A,1.00,0,0,0,NaN
4,512,139,F2-B,1.00,3,0,3,1.0
...,...,...,...,...,...,...,...,...
8725,1973,669,D0,0.00,0,0,0,NaN
8726,1973,669,F0,0.75,1,1,1,1.0
8727,1973,669,F1,0.50,0,0,0,NaN
8728,1973,669,F2-A,0.50,0,0,0,NaN


## 15. Paired paper-cluster bootstrap

Full FIT only: 10,000 resamples, paper_id, seed 42. F0/F1/F2 vs D0; F1/F2 vs F0; F2 vs F1. Every F2 candidate is reported; selected F2 is explicitly post-selection. Intervals are unadjusted; crossing zero establishes no superiority. Tiny smoke does not produce inferential comparisons.


In [13]:
show_csv('bootstrap.csv')


,comparison,actual_left,point_estimate,ci95_low,ci95_high,resamples,cluster,seed,post_selection,multiplicity_adjusted
0,F0 - D0,F0,0.004393,-0.008467,0.017454,10000,paper_id,42,False,False
1,F1 - D0,F1,0.001463,-0.002879,0.005921,10000,paper_id,42,False,False
2,F1 - F0,F1,-0.002930,-0.015381,0.009555,10000,paper_id,42,False,False
3,F2-A - D0,F2-A,0.003985,-0.000003,0.008224,10000,paper_id,42,False,False
4,F2-A - F0,F2-A,-0.000408,-0.012923,0.011982,10000,paper_id,42,False,False
5,F2-A - F1,F2-A,0.002522,-0.000838,0.006018,10000,paper_id,42,False,False
6,F2-B - D0,F2-B,0.001235,-0.004746,0.007241,10000,paper_id,42,False,False
7,F2-B - F0,F2-B,-0.003158,-0.015524,0.008825,10000,paper_id,42,False,False
8,F2-B - F1,F2-B,-0.000227,-0.005026,0.004358,10000,paper_id,42,False,False
9,F2-frozen - D0,F2-A,0.003985,-0.000003,0.008224,10000,paper_id,42,True,False


## 16. F2 variant freeze / 17. Intervention-mode freeze

Maximize FIT F1; retain candidates within 0.003. Preserve the existing cost-first convention: lower cumulative prompt tokens, then materially lower HURT tier (0.01 absolute difference), generated tokens, latency, shorter template and stable name. Freeze F2 first; then choose among F0, F1 and frozen F2. D0 is external SKIP. Smoke never freezes.


In [14]:
show_json('frozen_w9f_f2.json')
show_json('frozen_w9f_intervention.json')


{'schema': 'W9-F-F2-v1',
 'frozen': True,
 'split': 'fit',
 'selected_variant': 'F2-A',
 'prompt': {'messages': [{'role': 'user',
    'content': 'Answer the question using the supplied source evidence. Treat the source evidence as authoritative. The initial answer is only a starting point and may be incomplete or wrong. Preserve parts of the initial answer only when they remain supported by the evidence. Add missing information supported by the evidence, correct conflicting claims, and remove unsupported material when necessary. Do not invent facts. Output only the final concise answer, without reasoning or chain-of-thought.\n\nInitial answer: {initial_draft}\n\nAdditional evidence:\n{evidence}\n\nQuestion: {question}'}],
  'prompt_hash': '10b48d1969224dbf0f5d79d3bfd7844ab828f44e0ae533f77b8d5f7f16467795'},
 'selection_rule': 'FIT only: maximum F1, retain candidates within .003. Preserve existing global cost-first convention: lower cumulative prompt tokens, then materially lower HURT (a

{'schema': 'W9-F-intervention-v1',
 'frozen': True,
 'split': 'fit',
 'selected_intervention_mode': 'F0',
 'actual_policy': 'F0',
 'selected_f2_variant': 'F2-A',
 'evidence_policy': 'E3-k3',
 'selection_rule': 'FIT only: maximum F1, retain candidates within .003. Preserve existing global cost-first convention: lower cumulative prompt tokens, then materially lower HURT (at least .01 above minimum eligible HURT is the higher-risk tier), then lower generated tokens, lower recorded answer-stage latency, shorter template, stable policy name. D0 is excluded from intervention selection.',
 'all_fit_candidates': [{'policy': 'F0',
   'n_queries': 1746,
   'f1': 0.3900240164002956,
   'exact_match': 0.22623138602520046,
   'delta_f1': 0.004392839902436558,
   'delta_f1_vs_f0': 0.0,
   'help': 0.21649484536082475,
   'hurt': 0.20904925544100803,
   'answer_changed': 0.6134020618556701,
   'strict_preservation_rate': 0.3865979381443299,
   'normalized_preservation_rate': 0.40148911798396336,
   'n

## 18. W9-G handoff — training OFF

After complete FIT only, save frozen intervention identity, prompt/evidence hashes, per-query D0/intervention F1, costs and explicitly FIT-only HELP/HURT labels. These labels are not inference features. W9-G must wait because W9-F changes the intervention action's utility surface; E3's oracle was F0-based. This notebook never calls controller training.


In [15]:
handoff_path = RESULT_ROOT/'w9f_controller_handoff.json'
if handoff_path.exists():
    handoff = json.loads(handoff_path.read_text())
    display({k:v for k,v in handoff.items() if k not in ('queries','intervention_freeze')})
    print('FIT research rows:', len(handoff['queries']))
else:
    print('Handoff pending complete FIT W9-F; no controller was trained.')


{'schema': 'W9-F-controller-handoff-v1',
 'split': 'fit',
 'fit_research_labels_only': True,
 'not_inference_features': ['d0_f1',
  'intervention_f1',
  'delta_f1',
  'help',
  'hurt'],
 'controller_trained_or_run': False,
 'reason_g_waits': 'E3 oracle used F0; W9-F can change the action utility surface. G must consume this frozen mode, not hard-coded F1/G1.'}

FIT research rows: 1746


## 19. Interpretation

Compare quality, preservation, HURT rescue and HELP loss together. F0 may remain preferable; conservative F1 may sacrifice gain to preserve answers; F2 may or may not balance the tradeoff. Do not tune prompts again until F2 wins. A point estimate is not established superiority; confirmation must use an appropriate non-FIT split later. Exact source support/lexical overlap is not semantic entailment or causal evidence. No TEST claim.

Analysis-only command after full completion (no generator loading):

`./sara_env/bin/python -m src.week9.utilization_f.runner --config configs/week9/w9f_utilization.yaml --stage analyze --resume`
